## Imports and Data

In [1]:
%pip install -e "C:\Users\chdem\0UNIVERSIDAD\CIG\code\FCI-SF"

# -*- coding: utf-8 -*-
from pathlib import Path
import sys

# Reuse SRC_DIR already defined in the notebook if available; otherwise fall back.
try:
    SRC_DIR  # use existing variable
except NameError:
    try:
        SRC_DIR = Path(__file__).resolve().parents[2]
    except NameError:
        SRC_DIR = Path.cwd().resolve().parents[2]

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
# Ensure the package source folder is on sys.path (adds <repo>/src)
src_pkg = SRC_DIR / "src"
if str(src_pkg) not in sys.path:
    sys.path.insert(0, str(src_pkg))

from causaldiscovery.algorithms.FCI_SF import fci_sf
from causaldiscovery.CItest.noCache_CI_Test import myTest
from causallearn.graph.GeneralGraph import GeneralGraph 

path = Path(
    r"C:\Users\chdem\0UNIVERSIDAD\CIG\code\FCI-SF\src\experiments\real\13059_2004_896_MOESM1_ESM.txt"
)

ERROR: file:///C:/Users/chdem/0UNIVERSIDAD/CIG/code/FCI-SF does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


Obtaining file:///C:/Users/chdem/0UNIVERSIDAD/CIG/code/FCI-SF
Note: you may need to restart the kernel to use updated packages.


c:\Users\chdem\anaconda3\envs\FCI-FS_env_v2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw = pd.read_csv(
    path,
    sep=r"\s+",
    skiprows=6,
)

# Gene-level annotations: do not use these directly as causal variables.
gene_metadata = raw[
    ["Pathwayname", "ECID", "AGI", "Genename", "Name", "Probeset"]
].copy()

# Extract c1, ..., c118.
expression = raw.filter(regex=r"^c\d+$").astype(float)

# Transpose: arrays become observations and genes become variables.
X = expression.T

# AGI identifiers are safer variable names than the informal gene names.
X.columns = gene_metadata["Genename"].str.upper().to_numpy()
X.index.name = "array"

print(raw.shape)  # (39, 124)
print(X.shape)    # (118, 39)

X_log = np.log2(X + 1)

X_standardized = (
    X_log - X_log.mean(axis=0)
) / X_log.std(axis=0, ddof=1)

assert X_standardized.shape == (118, 39)
assert not X_standardized.isna().any().any()

data = X_log.to_numpy()
variable_names = X_log.columns.tolist()

(39, 124)
(118, 39)


## Causal Discovery with FCI-SF

In [3]:
print(variable_names)



['AACT1', 'AACT2', 'CMK', 'DPPS1', 'DPPS2', 'DPPS3', 'DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'FPPS1', 'FPPS2', 'GGPPS1MT', 'GGPPS2', 'GGPPS3', 'GGPPS4', 'GGPPS5', 'GGPPS6', 'GGPPS8', 'GGPPS9', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'GPPS', 'HDR', 'HDS', 'HMGR1', 'HMGR2', 'HMGS', 'IPPI1', 'IPPI2', 'MCT', 'MECPS', 'MK', 'MPDC1', 'MPDC2', 'PPDS1', 'PPDS2MT', 'UPPS1']


In [4]:
from IPython.display import HTML, display
from causallearn.utils.GraphUtils import GraphUtils


# Manually reconstructed from the layout in the original figure.
# Coordinates are arbitrary Graphviz coordinates, not image pixels.
WILLE_POSITIONS = {
    # --------------------------------------------------------------
    # MEP / non-mevalonate pathway: left
    # --------------------------------------------------------------
    "DXPS1":        (0.7, 9.4),
    "DXPS2(CLA1)":  (2.0, 9.4),
    "DXPS3":        (3.3, 9.4),

    "DXR":          (2.0, 8.5),
    "MCT":          (2.0, 7.7),
    "CMK":          (2.0, 6.9),
    "MECPS":        (2.0, 6.1),
    "HDS":          (2.0, 5.3),
    "HDR":          (2.0, 4.5),
    "IPPI1":        (2.0, 3.7),

    "GPPS":         (0.9, 2.7),

    # The paper groups these six genes in one node.
    "GGPPS2":       (0.2, 1.8),
    "GGPPS6":       (1.1, 1.8),
    "GGPPS8":       (2.0, 1.8),
    "GGPPS10":      (0.2, 0.9),
    "GGPPS11":      (1.1, 0.9),
    "GGPPS12":      (2.0, 0.9),

    "PPDS1":        (3.0, 1.8),
    "PPDS2MT":      (4.0, 1.8),

    # --------------------------------------------------------------
    # Mitochondrial group: centre
    # --------------------------------------------------------------
    "UPPS1":        (5.7, 2.8),
    "DPPS2":        (5.7, 1.9),

    # The paper groups GGPPS1, GGPPS5 and GGPPS9.
    "GGPPS1MT":     (4.8, 0.9),
    "GGPPS5":       (5.7, 0.9),
    "GGPPS9":       (6.6, 0.9),

    # --------------------------------------------------------------
    # MVA / mevalonate pathway: right
    # --------------------------------------------------------------
    "AACT1":        (7.8, 9.4),
    "AACT2":        (9.7, 9.4),

    "HMGS":         (8.75, 8.3),

    "HMGR1":        (7.8, 7.2),
    "HMGR2":        (9.7, 7.2),

    "MK":           (8.75, 6.1),

    "MPDC1":        (7.8, 5.0),
    "MPDC2":        (9.7, 5.0),

    "IPPI2":        (8.75, 3.7),

    "FPPS1":        (8.6, 2.6),
    "FPPS2":        (10.1, 2.6),

    # The paper groups DPPS1 and DPPS3.
    "DPPS1":        (7.6, 1.1),
    "DPPS3":        (8.5, 1.1),

    # The paper groups GGPPS3 and GGPPS4.
    "GGPPS3":       (9.6, 1.1),
    "GGPPS4":       (10.6, 1.1),
}

In [5]:
def draw_wille_graph(
    graph,
    variable_names,
    width="100%",
    max_height=700,
):
    variable_names = list(variable_names)

    missing_positions = [
        name for name in variable_names
        if name not in WILLE_POSITIONS
    ]

    if missing_positions:
        raise ValueError(
            "No fixed position defined for: "
            + ", ".join(missing_positions)
        )

    # Preserve the causal-learn edge endpoint representation.
    pyd = GraphUtils.to_pydot(
        graph,
        labels=variable_names,
    )

    # General graph settings.
    pyd.set_overlap("true")
    pyd.set_splines("true")
    pyd.set_outputorder("edgesfirst")
    pyd.set_margin("0")
    pyd.set_pad("0.15")

    # GraphUtils uses integer node IDs 0, 1, ..., while displaying
    # variable_names as labels.
    #
    # Iterating over all nodes also handles causal-learn versions that
    # produce duplicate pydot node declarations.
    for node in pyd.get_nodes():
        node_id = node.get_name().strip('"')

        if not node_id.isdigit():
            continue

        index = int(node_id)

        if index >= len(variable_names):
            continue

        variable = variable_names[index]
        x, y = WILLE_POSITIONS[variable]

        # The exclamation mark makes the position fixed under neato.
        node.set_pos(f'"{x},{y}!"')
        node.set_pin("true")

        node.set_shape("box")
        node.set_style('"rounded,filled"')
        node.set_fillcolor("white")
        node.set_fontsize("9")
        node.set_margin('"0.06,0.035"')
        node.set_penwidth("1")

    # Render directly in memory using neato.
    svg = pyd.create_svg(prog="neato").decode("utf-8")

    # Override the physical dimensions inserted by Graphviz so that the
    # result fits inside the notebook.
    svg = svg.replace(
        "<svg ",
        (
            f'<svg style="width:{width}; '
            f'max-height:{max_height}px; '
            f'height:auto;" '
        ),
        1,
    )

    display(
        HTML(
            f"""
            <div style="
                width: 100%;
                overflow: auto;
                text-align: center;
            ">
                {svg}
            </div>
            """
        )
    )

In [6]:

CI_test = myTest(X_standardized)
ALPHA = 0.01

fci_stable_full = fci_sf(data, independence_test_method=CI_test,
 initial_sep_sets = {}, alpha= ALPHA,  initial_graph = GeneralGraph([]),
  new_node_names = variable_names, verbose = False)

draw_wille_graph(
    fci_stable_full[0],
    variable_names,
    width="100%",
    max_height=650,
)

AACT2 --> HMGR2
AACT2 --> MK
MPDC1 --> AACT2
PPDS1 --> DPPS2
UPPS1 --> DXR
HMGS --> FPPS1
MK --> FPPS2
FPPS2 --> MPDC1


In [7]:
## Incremental Learning with a Causal Order

In [8]:
names_1 = ["DXPS1", "DXPS2(CLA1)", "DXPS3", "DXR", "AACT1", "AACT2", "HMGS"]

names_2 = ["HMGR1", "HMGR2", "MK", "MPDC1", "MPDC2", "MCT", "CMK", "MECPS", "HDS"] 
           
names_3 = ["IPPI1", "IPPI2", "FPPS1", "FPPS2", "HDR",  "GPPS", "PPDS1", "PPDS2MT"]

names_4 = ["GGPPS2", "GGPPS6", "GGPPS8", "GGPPS10", "GGPPS11", "GGPPS12", "DPPS1",  "DPPS3",  "GGPPS3", "GGPPS4"]

names_5 = ["UPPS1", "DPPS2","GGPPS1MT", "GGPPS5", "GGPPS9"]


In [9]:
from collections import Counter
from collections.abc import Set

import pydot
from IPython.display import HTML, display


# (graph: Graph, independence_test_method: CIT,  node1: Node, node2: Node, edge: Edge, alpha: float, sep_sets: Dict[Tuple[int, int], Set[int]], old_nodes = None 

def learn_and_draw(feature_names, new_node_names, initial_sep_sets= None, initial_graph: GeneralGraph = None):
    """
    Learn an FCI-SF graph over a selected set of genes and draw it using
    the positions from the original Wille et al. figure.

    Parameters
    ----------
    feature_names : list[str]
        Gene names to include in the causal-discovery analysis.

    Returns
    -------
    graph : causallearn.graph.GeneralGraph.GeneralGraph
        The learned graph.
    """


    # --------------------------------------------------------------
    # 1. Normalize and validate names
    # --------------------------------------------------------------
    names = [
        str(name).strip().upper()
        for name in feature_names
    ]

    duplicates = [
        name
        for name, count in Counter(names).items()
        if count > 1
    ]

    if duplicates:
        raise ValueError(
            "Repeated feature names: " + ", ".join(duplicates)
        )

    missing_data = [
        name
        for name in names
        if name not in X_standardized.columns
    ]

    if missing_data:
        raise ValueError(
            "Features not found in X_standardized: "
            + ", ".join(missing_data)
        )

    missing_positions = [
        name
        for name in names
        if name not in WILLE_POSITIONS
    ]

    if missing_positions:
        raise ValueError(
            "Features without a position in WILLE_POSITIONS: "
            + ", ".join(missing_positions)
        )

    if len(names) < 2:
        raise ValueError(
            "At least two features are required."
        )

    print(f"Selected features: {', '.join(names)}")
    # --------------------------------------------------------------
    # 2. Select data
    # --------------------------------------------------------------
    X_subset = X_standardized.loc[:, names].copy()
    data_subset = X_subset.to_numpy()

    print(f"Number of observations: {X_subset.shape[0]}")
    print(f"Number of variables: {X_subset.shape[1]}")
    print("Variables:", names)

    # --------------------------------------------------------------
    # 3. Learn graph
    # --------------------------------------------------------------
    # Replace this line with the exact fci_sf call that is already
    # working in your notebook.
    print("New node names:", new_node_names)

    fci_sf_output = fci_sf(data_subset, independence_test_method=CI_test,
        initial_sep_sets = initial_sep_sets, alpha= ALPHA,  initial_graph = initial_graph,
        new_node_names = new_node_names, verbose = False)
    print("FCI-SF output:", fci_sf_output)
    graph = fci_sf_output[0]
    sepsets =  fci_sf_output[5]

    # --------------------------------------------------------------
    # 4. Print learned edges
    # --------------------------------------------------------------
    edges = graph.get_graph_edges()

    print("\nLearned graph:")

    if not edges:
        print("No edges were found.")
    else:
        for edge in edges:
            print(edge)

    # --------------------------------------------------------------
    # 5. Convert GeneralGraph to pydot
    # --------------------------------------------------------------
    pyd = GraphUtils.to_pydot(
        graph,
        labels=names,
    )

    pyd.set_overlap("true")
    pyd.set_splines("true")
    pyd.set_outputorder("edgesfirst")
    pyd.set_margin("0")
    pyd.set_pad("0.15")

    # --------------------------------------------------------------
    # 6. Assign the original-paper positions
    # --------------------------------------------------------------
    for node in pyd.get_nodes():
        node_id = node.get_name().strip('"')

        # GraphUtils normally names nodes 0, 1, 2, ...
        if not node_id.isdigit():
            continue

        node_index = int(node_id)

        if node_index >= len(names):
            continue

        feature = names[node_index]
        x, y = WILLE_POSITIONS[feature]

        node.set_pos(f'"{x},{y}!"')
        node.set_pin("true")
        node.set_shape("box")
        node.set_style('"rounded,filled"')
        node.set_fillcolor("white")
        node.set_fontsize("10")
        node.set_margin('"0.07,0.04"')
        node.set_penwidth("1")

    # --------------------------------------------------------------
    # 7. Add invisible anchors
    #
    # These preserve the complete coordinate system of the original
    # figure. Unselected genes therefore leave blank spaces rather
    # than causing the selected nodes to be recentered.
    # --------------------------------------------------------------
    anchor_positions = {
        "__bottom_left":  (-0.3, 0.3),
        "__bottom_right": (11.0, 0.3),
        "__top_left":     (-0.3, 9.9),
        "__top_right":    (11.0, 9.9),
    }

    for anchor_name, (x, y) in anchor_positions.items():
        anchor = pydot.Node(
            anchor_name,
            label="",
            shape="point",
            width="0",
            height="0",
            style="invis",
            pos=f'"{x},{y}!"',
            pin="true",
        )
        pyd.add_node(anchor)

    # --------------------------------------------------------------
    # 8. Render directly in the notebook
    # --------------------------------------------------------------
    svg = pyd.create_svg(prog="neato").decode("utf-8")

    svg = svg.replace(
        "<svg ",
        (
            '<svg style="'
            'width:100%; '
            'max-width:950px; '
            'height:auto;'
            '" '
        ),
        1,
    )

    display(
        HTML(
            f"""
            <div style="
                width: 100%;
                max-height: 700px;
                overflow: auto;
                text-align: center;
            ">
                {svg}
            </div>
            """
        )
    )

    return graph, sepsets

In [10]:
graph_1, sepsets_1 = learn_and_draw(names_1, new_node_names=names_1)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS
Number of observations: 118
Number of variables: 7
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS']
New node names: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS']
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CDF377E80>, 36, 0.3888888888888889, 0.014022588729858398, [<causallearn.graph.Edge.Edge object at 0x0000024CDF377E20>, <causallearn.graph.Edge.Edge object at 0x0000024CDF377370>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3774C0>, <causallearn.graph.Edge.Edge object at 0x0000024CDF376F80>], {(0, 3): set(), (3, 0): set(), (0, 4): set(), (4, 0): set(), (0, 5): set(), (5, 0): set(), (0, 6): set(), (6, 0): set(), (1, 2): set(), (2, 1): set(), (1, 5): set(), (5, 1): set(), (1, 6): set(), (6, 1): set(), (2, 3): set(), (3, 2): set(), (2, 4): set(), (4, 2): set(), (2, 5): set(), (5, 2): set(), (2, 6): set(), (6, 2): set(), (3, 4):

In [11]:
graph_2, sepsets_2 = learn_and_draw(names_1 + names_2, new_node_names=names_2, initial_sep_sets=sepsets_1, initial_graph=graph_1)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS
Number of observations: 118
Number of variables: 16
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS']
New node names: ['HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS']
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CDF377E80>, 219, 0.4474885844748858, 0.06402277946472168, [<causallearn.graph.Edge.Edge object at 0x0000024CDF376E00>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3779D0>, <causallearn.graph.Edge.Edge object at 0x0000024CDF375690>, <causallearn.graph.Edge.Edge object at 0x0000024CDF376E30>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3756C0>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3747F0>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3747C0>, <causallearn.graph.Edge.Ed

In [12]:
graph_3, sepsets_3 = learn_and_draw(names_1 + names_2 + names_3, new_node_names=names_3, initial_sep_sets=sepsets_2, initial_graph=graph_2)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS, IPPI1, IPPI2, FPPS1, FPPS2, HDR, GPPS, PPDS1, PPDS2MT
Number of observations: 118
Number of variables: 24
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS', 'IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT']
New node names: ['IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT']
MK --> GPPS
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CDF377E80>, 448, 0.6808035714285714, 0.13762784004211426, [<causallearn.graph.Edge.Edge object at 0x0000024CBB31BE80>, <causallearn.graph.Edge.Edge object at 0x0000024CDF377E20>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3755A0>, <causallearn.graph.Edge.Edge object at 0x0000024CDF377A60>, <causallearn.graph.Edge.Edge object at 0x0000024CDF375570>, <causalle

In [13]:
graph_4, sepsets_4 = learn_and_draw(names_1 + names_2 + names_3 + names_4, new_node_names=names_4, initial_sep_sets=sepsets_3, initial_graph=graph_3)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS, IPPI1, IPPI2, FPPS1, FPPS2, HDR, GPPS, PPDS1, PPDS2MT, GGPPS2, GGPPS6, GGPPS8, GGPPS10, GGPPS11, GGPPS12, DPPS1, DPPS3, GGPPS3, GGPPS4
Number of observations: 118
Number of variables: 34
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS', 'IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT', 'GGPPS2', 'GGPPS6', 'GGPPS8', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'DPPS1', 'DPPS3', 'GGPPS3', 'GGPPS4']
New node names: ['GGPPS2', 'GGPPS6', 'GGPPS8', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'DPPS1', 'DPPS3', 'GGPPS3', 'GGPPS4']
DXPS1 --> GGPPS8
GGPPS3 --> DXPS1
GGPPS6 --> MK
GGPPS3 --> GGPPS6
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CDF377E80>, 1933, 0.9596482152095189, 0.5114114284515381, [<causallearn.graph.Edge.Edge object at 0x00

In [14]:
graph_5, sepsets_5 = learn_and_draw(names_1 + names_2 + names_3 + names_4 + names_5, new_node_names=names_5, initial_sep_sets=sepsets_4, initial_graph=graph_4)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS, IPPI1, IPPI2, FPPS1, FPPS2, HDR, GPPS, PPDS1, PPDS2MT, GGPPS2, GGPPS6, GGPPS8, GGPPS10, GGPPS11, GGPPS12, DPPS1, DPPS3, GGPPS3, GGPPS4, UPPS1, DPPS2, GGPPS1MT, GGPPS5, GGPPS9
Number of observations: 118
Number of variables: 39
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS', 'IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT', 'GGPPS2', 'GGPPS6', 'GGPPS8', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'DPPS1', 'DPPS3', 'GGPPS3', 'GGPPS4', 'UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9']
New node names: ['UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9']
DXPS1 --> GGPPS8
GGPPS3 --> DXPS1
GGPPS6 --> MK
MK --> GGPPS9
MPDC2 --> UPPS1
GGPPS3 --> GGPPS6
GGPPS1MT --> GGPPS5
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CDF377E80>, 1

## Other Order


In [15]:
graph_6, sepsets_6 = learn_and_draw(names_1 + names_2, new_node_names=names_1 + names_2)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS
Number of observations: 118
Number of variables: 16
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS']
New node names: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS']
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CDF3776A0>, 252, 0.42063492063492064, 0.07568717002868652, [<causallearn.graph.Edge.Edge object at 0x0000024CBB31AB60>, <causallearn.graph.Edge.Edge object at 0x0000024CDF374F10>, <causallearn.graph.Edge.Edge object at 0x0000024CDF374C70>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3771F0>, <causallearn.graph.Edge.Edge object at 0x0000024CDF377850>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3774C0>, <causallearn.graph.

In [16]:
graph_7, sepsets_7 = learn_and_draw(names_1 + names_2 + names_3 + names_4, new_node_names=names_3 + names_4, initial_sep_sets=sepsets_6, initial_graph=graph_6)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS, IPPI1, IPPI2, FPPS1, FPPS2, HDR, GPPS, PPDS1, PPDS2MT, GGPPS2, GGPPS6, GGPPS8, GGPPS10, GGPPS11, GGPPS12, DPPS1, DPPS3, GGPPS3, GGPPS4
Number of observations: 118
Number of variables: 34
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS', 'IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT', 'GGPPS2', 'GGPPS6', 'GGPPS8', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'DPPS1', 'DPPS3', 'GGPPS3', 'GGPPS4']
New node names: ['IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT', 'GGPPS2', 'GGPPS6', 'GGPPS8', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'DPPS1', 'DPPS3', 'GGPPS3', 'GGPPS4']
GGPPS6 --> MK
GGPPS3 --> GGPPS6
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CDF3776A0>, 2459, 0.8906059373729158, 0.7270822525024414, [<causa

In [17]:
graph_8, sepsets_8 = learn_and_draw(names_1 + names_2 + names_3 + names_4 + names_5, new_node_names= names_5, initial_sep_sets=sepsets_7, initial_graph=graph_7)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS, IPPI1, IPPI2, FPPS1, FPPS2, HDR, GPPS, PPDS1, PPDS2MT, GGPPS2, GGPPS6, GGPPS8, GGPPS10, GGPPS11, GGPPS12, DPPS1, DPPS3, GGPPS3, GGPPS4, UPPS1, DPPS2, GGPPS1MT, GGPPS5, GGPPS9
Number of observations: 118
Number of variables: 39
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS', 'IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT', 'GGPPS2', 'GGPPS6', 'GGPPS8', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'DPPS1', 'DPPS3', 'GGPPS3', 'GGPPS4', 'UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9']
New node names: ['UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9']
GGPPS6 --> MK
MK --> GGPPS9
MPDC2 --> UPPS1
GGPPS3 --> GGPPS6
GGPPS1MT --> GGPPS5
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CDF3776A0>, 1031, 0.8874878758486906, 0.4511218

## Third Attemp Trying a Different Order

In [18]:
graph_9, sepsets_9 = learn_and_draw(names_1  + names_5, new_node_names= names_1  + names_5)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, UPPS1, DPPS2, GGPPS1MT, GGPPS5, GGPPS9
Number of observations: 118
Number of variables: 12
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9']
New node names: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9']
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CBB39CF70>, 186, 0.521505376344086, 0.06451797485351562, [<causallearn.graph.Edge.Edge object at 0x0000024CBB31AB60>, <causallearn.graph.Edge.Edge object at 0x0000024CBB31B430>, <causallearn.graph.Edge.Edge object at 0x0000024CBB357160>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39FD60>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39C130>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39FDC0>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39C310>, <causallearn.graph.

In [19]:
graph_10, sepsets_10 = learn_and_draw(names_1  + names_5 + names_2, new_node_names= names_2, initial_sep_sets=sepsets_9, initial_graph=graph_9)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, UPPS1, DPPS2, GGPPS1MT, GGPPS5, GGPPS9, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS
Number of observations: 118
Number of variables: 21
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS']
New node names: ['HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS']
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CBB39CF70>, 264, 0.4090909090909091, 0.10055088996887207, [<causallearn.graph.Edge.Edge object at 0x0000024CBB357160>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39CB20>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39D540>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39DED0>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39C9D0>, <causallearn.graph.Edge.Edge object at 0x0000024CBB39CAF

In [20]:
graph_11, sepsets_11 = learn_and_draw(names_1  + names_5 + names_2 + names_3, new_node_names= names_3, initial_sep_sets=sepsets_10, initial_graph=graph_10)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, UPPS1, DPPS2, GGPPS1MT, GGPPS5, GGPPS9, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS, IPPI1, IPPI2, FPPS1, FPPS2, HDR, GPPS, PPDS1, PPDS2MT
Number of observations: 118
Number of variables: 29
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS', 'IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT']
New node names: ['IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT']
AACT1 --> GGPPS5
GGPPS5 --> PPDS2MT
FCI-SF output: (<causallearn.graph.GeneralGraph.GeneralGraph object at 0x0000024CBB39CF70>, 977, 0.8075742067553736, 0.3022885322570801, [<causallearn.graph.Edge.Edge object at 0x0000024CDF374580>, <causallearn.graph.Edge.Edge object at 0x0000024CDF376B60>, <causallearn.graph.Edge.Edge object at 0x0000024CDF3762C0>, <causallearn.gra

In [21]:
graph_12, sepsets_12 = learn_and_draw(names_1  + names_5 + names_2 + names_3 + names_4, new_node_names= names_4, initial_sep_sets=sepsets_11, initial_graph=graph_11)

Selected features: DXPS1, DXPS2(CLA1), DXPS3, DXR, AACT1, AACT2, HMGS, UPPS1, DPPS2, GGPPS1MT, GGPPS5, GGPPS9, HMGR1, HMGR2, MK, MPDC1, MPDC2, MCT, CMK, MECPS, HDS, IPPI1, IPPI2, FPPS1, FPPS2, HDR, GPPS, PPDS1, PPDS2MT, GGPPS2, GGPPS6, GGPPS8, GGPPS10, GGPPS11, GGPPS12, DPPS1, DPPS3, GGPPS3, GGPPS4
Number of observations: 118
Number of variables: 39
Variables: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9', 'HMGR1', 'HMGR2', 'MK', 'MPDC1', 'MPDC2', 'MCT', 'CMK', 'MECPS', 'HDS', 'IPPI1', 'IPPI2', 'FPPS1', 'FPPS2', 'HDR', 'GPPS', 'PPDS1', 'PPDS2MT', 'GGPPS2', 'GGPPS6', 'GGPPS8', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'DPPS1', 'DPPS3', 'GGPPS3', 'GGPPS4']
New node names: ['GGPPS2', 'GGPPS6', 'GGPPS8', 'GGPPS10', 'GGPPS11', 'GGPPS12', 'DPPS1', 'DPPS3', 'GGPPS3', 'GGPPS4']
DXPS2(CLA1) --> PPDS1
DXPS2(CLA1) --> GGPPS11
GGPPS12 --> DXPS2(CLA1)
GGPPS4 --> GGPPS1MT
GGPPS5 --> PPDS2MT
GGPPS11 --> GGPPS9
GGPPS9 --> GGPPS12
DPPS3 --> G

## Bootstraping


In [22]:


ALPHA = 0.05

def learn_incremental_stage(
    X_data,
    feature_names,
    new_node_names,
    independence_test_method,
    initial_sep_sets=None,
    initial_graph=None,
):
    names = [
        str(name).strip().upper()
        for name in feature_names
    ]

    new_names = [
        str(name).strip().upper()
        for name in new_node_names
    ]

    missing = [
        name
        for name in names
        if name not in X_data.columns
    ]

    if missing:
        raise ValueError(
            "Features not found in the dataset: "
            + ", ".join(missing)
        )

    X_subset = X_data.loc[:, names]
    data_subset = X_subset.to_numpy(dtype=float)

    output = fci_sf(
        data_subset,
        independence_test_method=independence_test_method,
        initial_sep_sets=initial_sep_sets,
        alpha=ALPHA,
        initial_graph=initial_graph,
        new_node_names=new_names,
        verbose=False,
    )

    return output[0], output[5]

In [23]:
from collections import Counter

import numpy as np


def bootstrap_incremental_fci_sf(
    feature_groups,
    n_bootstraps=500,
    random_state=None,
    store_sep_sets=False,
    verbose=True,
):
    """
    Bootstrap the incremental FCI-SF procedure.

    For each bootstrap sample, four graphs are learned incrementally.
    Only each GeneralGraph.graph NumPy matrix is stored.

    Parameters
    ----------
    feature_groups : sequence[sequence[str]]
        Ordered groups such as:
        [names_1, names_2, names_3, names_4].

    n_bootstraps : int
        Number of bootstrap repetitions.

    random_state : int or None
        Random seed.

    store_sep_sets : bool
        Whether to store the separating sets returned at every stage.

    verbose : bool
        Whether to print progress.

    Returns
    -------
    graph_matrices : numpy.ndarray, dtype=object
        Shape (n_stages, n_bootstraps).

        graph_matrices[s, b] is an independent NumPy copy of the
        GeneralGraph.graph matrix learned at stage s of bootstrap b.

    bootstrap_indices : numpy.ndarray
        Shape (n_bootstraps, n_observations).

    stage_names : list[list[str]]
        Variable order corresponding to each stage's graph matrices.

    sep_sets_matrix : numpy.ndarray, optional
        Returned when store_sep_sets=True.
    """
    if n_bootstraps < 1:
        raise ValueError("n_bootstraps must be at least 1.")

    if not feature_groups:
        raise ValueError("feature_groups cannot be empty.")

    # --------------------------------------------------------------
    # Normalize and validate feature groups
    # --------------------------------------------------------------
    groups = [
        [
            str(name).strip().upper()
            for name in group
        ]
        for group in feature_groups
    ]

    if any(len(group) == 0 for group in groups):
        raise ValueError("Feature groups cannot be empty.")

    all_names = [
        name
        for group in groups
        for name in group
    ]

    duplicates = [
        name
        for name, count in Counter(all_names).items()
        if count > 1
    ]

    if duplicates:
        raise ValueError(
            "Features occur in more than one group: "
            + ", ".join(duplicates)
        )

    missing = [
        name
        for name in all_names
        if name not in X_standardized.columns
    ]

    if missing:
        raise ValueError(
            "Features not found in X_standardized: "
            + ", ".join(missing)
        )

    # Variable order corresponding to each stage matrix.
    stage_names = []
    cumulative_names = []

    for group in groups:
        cumulative_names = cumulative_names + group
        stage_names.append(cumulative_names.copy())

    # --------------------------------------------------------------
    # Allocate results
    # --------------------------------------------------------------
    n_stages = len(groups)
    n_observations = X_standardized.shape[0]

    graph_matrices = np.empty(
        (n_stages, n_bootstraps),
        dtype=object,
    )

    bootstrap_indices = np.empty(
        (n_bootstraps, n_observations),
        dtype=int,
    )

    if store_sep_sets:
        sep_sets_matrix = np.empty(
            (n_stages, n_bootstraps),
            dtype=object,
        )

    rng = np.random.default_rng(random_state)

    if verbose:
        print(f"Observations: {n_observations}")
        print(f"Bootstrap repetitions: {n_bootstraps}")
        print(
            "Variables per stage:",
            [len(names) for names in stage_names],
        )

    # --------------------------------------------------------------
    # Bootstrap repetitions
    # --------------------------------------------------------------
    for bootstrap_id in range(n_bootstraps):

        # ----------------------------------------------------------
        # One row bootstrap sample for the entire incremental chain
        # ----------------------------------------------------------
        indices = rng.integers(
            low=0,
            high=n_observations,
            size=n_observations,
        )

        bootstrap_indices[bootstrap_id] = indices

        # All features in the exact incremental order:
        # names_1 + names_2 + names_3 + names_4
        X_bootstrap = (
            X_standardized
            .iloc[indices]
            .loc[:, all_names]
            .reset_index(drop=True)
        )

        # One CI-test object for this complete bootstrap dataset.
        bootstrap_ci_test = myTest(
            X_bootstrap.to_numpy(dtype=float)
        )

        previous_graph = None
        previous_sep_sets = None

        # ----------------------------------------------------------
        # Four incremental stages using the same CI-test object
        # ----------------------------------------------------------
        for stage_id, new_names in enumerate(groups):

            graph_stage, sep_sets_stage = learn_incremental_stage(
                X_data=X_bootstrap,
                feature_names=stage_names[stage_id],
                new_node_names=new_names,
                independence_test_method=bootstrap_ci_test,
                initial_sep_sets=previous_sep_sets,
                initial_graph=previous_graph,
            )

            graph_matrices[
                stage_id,
                bootstrap_id
            ] = graph_stage.graph.copy()

            previous_graph = graph_stage
            previous_sep_sets = sep_sets_stage

        if verbose:
            completed = bootstrap_id + 1

            if (
                completed == 1
                or completed % 10 == 0
                or completed == n_bootstraps
            ):
                edge_counts = []

                for stage_id in range(n_stages):
                    matrix = graph_matrices[
                        stage_id,
                        bootstrap_id
                    ]

                    # Each adjacency is represented twice in the matrix.
                    n_edges = np.count_nonzero(
                        np.triu(matrix != 0, k=1)
                    )

                    edge_counts.append(int(n_edges))

                print(
                    f"Bootstrap {completed}/{n_bootstraps} "
                    f"completed. Edge counts: {edge_counts}"
                )

    if store_sep_sets:
        return (
            graph_matrices,
            bootstrap_indices,
            stage_names,
            sep_sets_matrix,
        )

    return graph_matrices, bootstrap_indices, stage_names


In [24]:
feature_groups = [
    names_1+names_5,
    names_2,
    names_3,
    names_4,
]

(
    bootstrap_graph_matrices,
    bootstrap_indices,
    bootstrap_stage_names,
) = bootstrap_incremental_fci_sf(
    feature_groups=feature_groups,
    n_bootstraps=500,
    random_state=123,
    verbose=True,
)

Observations: 118
Bootstrap repetitions: 500
Variables per stage: [12, 21, 29, 39]
DXPS2(CLA1) --> DXR
DXR --> UPPS1
DXR --> HDS
MECPS --> AACT1
MECPS --> CMK
HDS --> MECPS
DXPS2(CLA1) --> HDR
DXR --> UPPS1
DXR --> HDS
MECPS --> AACT1
HMGS --> FPPS1
DPPS2 --> HDS
PPDS1 --> DPPS2
MECPS --> CMK
HDS --> MECPS
FPPS1 --> IPPI2
HDR --> PPDS1
PPDS1 --> PPDS2MT
DXPS2(CLA1) --> HDR
DXR --> UPPS1
DXR --> HDS
MECPS --> AACT1
HMGS --> FPPS1
DPPS2 --> HDS
PPDS1 --> DPPS2
MECPS --> CMK
HDS --> MECPS
FPPS1 --> IPPI2
HDR --> PPDS1
Bootstrap 1/500 completed. Edge counts: [10, 22, 32, 51]
DXR --> HDS
MCT --> CMK
DXR --> HDS
HMGS --> FPPS1
MCT --> CMK
DXR --> HDS
HMGS --> FPPS1
DPPS2 --> HMGR2
PPDS1 --> DPPS2
MCT --> CMK
PPDS1 --> HDR
DXR --> UPPS1
MCT --> DXR
HMGR2 --> HDS
HMGS --> IPPI2
HMGS --> FPPS1
HMGR2 --> HDS
MPDC1 --> FPPS2
MECPS --> AACT1
AACT2 --> HMGR2
MK --> AACT2
AACT2 --> MPDC1
AACT2 --> GGPPS8
FPPS1 --> HMGS
HMGR2 --> HDS
MPDC1 --> FPPS2
MECPS --> CMK
HDS --> MECPS
HMGS --> MPDC2
MPDC2 --

## Get a Graph from Bootstrap

In [25]:
from dataclasses import dataclass
from typing import Dict, List, Sequence

import numpy as np
import pydot
from IPython.display import HTML, display

TAIL = -1
NULL = 0
ARROW = 1
CIRCLE = 2
STAR = 3
TAIL_AND_ARROW = 4
ARROW_AND_ARROW = 5


ENDPOINT_CODES = {
    "TAIL": TAIL,
    "ARROW": ARROW,
    "STAR": STAR,
    "TAIL_AND_ARROW": TAIL_AND_ARROW,
    "ARROW_AND_ARROW": ARROW_AND_ARROW,
}

In [26]:
@dataclass
class BootstrapStageSummary:
    """
    Bootstrap counts for one incremental stage.
    """

    feature_names: List[str]
    n_bootstraps: int

    # Symmetric matrix
    adjacency_counts: np.ndarray

    # Five nonsymmetric matrices
    endpoint_counts: Dict[str, np.ndarray]

In [27]:
from typing import List

from pydot import Sequence


def aggregate_bootstrap_matrices(
    bootstrap_graph_matrices: np.ndarray,
    stage_names: Sequence[Sequence[str]],
) -> List[BootstrapStageSummary]:
    """
    Aggregate bootstrap GeneralGraph.graph matrices separately by stage.

    Parameters
    ----------
    bootstrap_graph_matrices
        Object array with shape:

            (n_stages, n_bootstraps)

        Each entry is a NumPy matrix copied from GeneralGraph.graph.

    stage_names
        Variable names corresponding to each stage matrix, in matrix order.

    Returns
    -------
    summaries
        One BootstrapStageSummary per incremental stage.
    """
    matrices = np.asarray(
        bootstrap_graph_matrices,
        dtype=object,
    )

    if matrices.ndim != 2:
        raise ValueError(
            "bootstrap_graph_matrices must have shape "
            "(n_stages, n_bootstraps)."
        )

    n_stages, n_bootstraps = matrices.shape

    if len(stage_names) != n_stages:
        raise ValueError(
            f"Expected {n_stages} stage-name lists, "
            f"received {len(stage_names)}."
        )

    summaries = []

    valid_values = {
        TAIL,
        NULL,
        ARROW,
        CIRCLE,
        STAR,
        TAIL_AND_ARROW,
        ARROW_AND_ARROW,
    }

    for stage_id in range(n_stages):
        names = [
            str(name).strip().upper()
            for name in stage_names[stage_id]
        ]

        expected_shape = (len(names), len(names))
        stage_matrix_list = []

        for bootstrap_id in range(n_bootstraps):
            matrix = np.asarray(
                matrices[stage_id, bootstrap_id],
                dtype=int,
            )

            if matrix.shape != expected_shape:
                raise ValueError(
                    f"Stage {stage_id + 1}, bootstrap "
                    f"{bootstrap_id + 1}: expected shape "
                    f"{expected_shape}, received {matrix.shape}."
                )

            observed_values = set(np.unique(matrix))

            invalid_values = observed_values - valid_values

            if invalid_values:
                raise ValueError(
                    f"Stage {stage_id + 1}, bootstrap "
                    f"{bootstrap_id + 1}: invalid endpoint values "
                    f"{sorted(invalid_values)}."
                )

            if np.any(np.diag(matrix) != NULL):
                raise ValueError(
                    f"Stage {stage_id + 1}, bootstrap "
                    f"{bootstrap_id + 1}: diagonal must be zero."
                )

            # A valid adjacency should be null on both sides or
            # non-null on both sides.
            null_pattern = matrix == NULL

            if not np.array_equal(
                null_pattern,
                null_pattern.T,
            ):
                raise ValueError(
                    f"Stage {stage_id + 1}, bootstrap "
                    f"{bootstrap_id + 1}: one-sided null endpoint "
                    "detected."
                )

            stage_matrix_list.append(matrix)

        # Shape:
        # (n_bootstraps, n_variables, n_variables)
        stage_stack = np.stack(
            stage_matrix_list,
            axis=0,
        )

        # ----------------------------------------------------------
        # Symmetric adjacency counts
        # ----------------------------------------------------------
        adjacency_present = (
            (stage_stack != NULL)
            | (stage_stack.transpose(0, 2, 1) != NULL)
        )

        adjacency_counts = adjacency_present.sum(
            axis=0,
            dtype=int,
        )

        np.fill_diagonal(adjacency_counts, 0)

        if not np.array_equal(
            adjacency_counts,
            adjacency_counts.T,
        ):
            raise RuntimeError(
                "The calculated adjacency-count matrix is not symmetric."
            )

        # ----------------------------------------------------------
        # Nonsymmetric endpoint counts
        # ----------------------------------------------------------
        endpoint_counts = {}

        for endpoint_name, endpoint_code in ENDPOINT_CODES.items():
            count_matrix = (
                stage_stack == endpoint_code
            ).sum(
                axis=0,
                dtype=int,
            )

            np.fill_diagonal(count_matrix, 0)

            endpoint_counts[endpoint_name] = count_matrix

        summaries.append(
            BootstrapStageSummary(
                feature_names=names,
                n_bootstraps=n_bootstraps,
                adjacency_counts=adjacency_counts,
                endpoint_counts=endpoint_counts,
            )
        )

    return summaries

In [28]:
print("Aggregating bootstrap matrices. bootstrap_graph_matrices shape:", bootstrap_graph_matrices.shape, "bootstrap_stage_names length:", len(bootstrap_stage_names))
print("First stage names:", bootstrap_stage_names[0])
print("First stage graph matrix shape:", bootstrap_graph_matrices[0, 0].shape)
bootstrap_summaries = aggregate_bootstrap_matrices(
    bootstrap_graph_matrices=bootstrap_graph_matrices,
    stage_names=bootstrap_stage_names,
)



Aggregating bootstrap matrices. bootstrap_graph_matrices shape: (4, 500) bootstrap_stage_names length: 4
First stage names: ['DXPS1', 'DXPS2(CLA1)', 'DXPS3', 'DXR', 'AACT1', 'AACT2', 'HMGS', 'UPPS1', 'DPPS2', 'GGPPS1MT', 'GGPPS5', 'GGPPS9']
First stage graph matrix shape: (12, 12)


In [29]:
stage_1 = bootstrap_summaries[3]

stage_1.adjacency_counts




array([[ 0,  1, 30, ...,  9,  8,  8],
       [ 1,  0,  6, ...,  0,  5,  0],
       [30,  6,  0, ...,  6, 21, 32],
       ...,
       [ 9,  0,  6, ...,  0, 73, 35],
       [ 8,  5, 21, ..., 73,  0, 19],
       [ 8,  0, 32, ..., 35, 19,  0]])

In [30]:
def _threshold_to_fraction(
    threshold: float,
    parameter_name: str,
) -> float:
    threshold = float(threshold)

    if 0 <= threshold <= 1:
        return threshold

    if 1 < threshold <= 100:
        return threshold / 100.0

    raise ValueError(
        f"{parameter_name} must be between 0 and 1, "
        "or between 0 and 100."
    )

In [31]:
def _select_consensus_endpoint(
    endpoint_counts: Dict[str, np.ndarray],
    i: int,
    j: int,
    denominator: int,
    endpoint_threshold: float,
) -> int:
    """
    Select the endpoint at node i for the pair (i, j).

    The most frequent non-circle endpoint is selected if it reaches
    the threshold. A tie at the highest count produces a circle.
    """
    if denominator <= 0:
        return CIRCLE

    candidates = []

    for endpoint_name, endpoint_code in ENDPOINT_CODES.items():
        count = int(
            endpoint_counts[endpoint_name][i, j]
        )

        frequency = count / denominator

        if frequency >= endpoint_threshold:
            candidates.append(
                (
                    count,
                    endpoint_code,
                    endpoint_name,
                )
            )

    if not candidates:
        return CIRCLE

    largest_count = max(
        count
        for count, _, _ in candidates
    )

    best_candidates = [
        candidate
        for candidate in candidates
        if candidate[0] == largest_count
    ]

    # Do not arbitrarily resolve equal bootstrap support.
    if len(best_candidates) > 1:
        return CIRCLE

    return best_candidates[0][1]

In [32]:
def build_consensus_matrices(
    summaries: Sequence[BootstrapStageSummary],
    adjacency_threshold: float = 0.5,
    endpoint_threshold: float = 0.5,
    endpoint_denominator: str = "all_graphs",
) -> List[np.ndarray]:
    """
    Build one bootstrap consensus graph matrix per incremental stage.

    Parameters
    ----------
    summaries
        Output from aggregate_bootstrap_matrices().

    adjacency_threshold
        Required adjacency frequency.

    endpoint_threshold
        Required endpoint frequency.

    endpoint_denominator
        "all_graphs":
            endpoint count / total number of bootstrap graphs.

        "adjacent_graphs":
            endpoint count / number of bootstrap graphs in which
            the pair was adjacent.

    Returns
    -------
    consensus_matrices
        List containing one integer graph matrix per stage.
    """
    adjacency_threshold = _threshold_to_fraction(
        adjacency_threshold,
        "adjacency_threshold",
    )

    endpoint_threshold = _threshold_to_fraction(
        endpoint_threshold,
        "endpoint_threshold",
    )

    valid_denominators = {
        "all_graphs",
        "adjacent_graphs",
    }

    if endpoint_denominator not in valid_denominators:
        raise ValueError(
            "endpoint_denominator must be either "
            "'all_graphs' or 'adjacent_graphs'."
        )

    consensus_matrices = []

    for summary in summaries:
        p = len(summary.feature_names)
        B = summary.n_bootstraps

        consensus = np.zeros(
            (p, p),
            dtype=int,
        )

        for i in range(p):
            for j in range(i + 1, p):
                adjacency_count = int(
                    summary.adjacency_counts[i, j]
                )

                adjacency_frequency = (
                    adjacency_count / B
                )

                if adjacency_frequency < adjacency_threshold:
                    continue

                if endpoint_denominator == "all_graphs":
                    endpoint_denominator_ij = B
                else:
                    endpoint_denominator_ij = adjacency_count

                # Endpoint at node i
                endpoint_i = _select_consensus_endpoint(
                    endpoint_counts=summary.endpoint_counts,
                    i=i,
                    j=j,
                    denominator=endpoint_denominator_ij,
                    endpoint_threshold=endpoint_threshold,
                )

                # Endpoint at node j
                endpoint_j = _select_consensus_endpoint(
                    endpoint_counts=summary.endpoint_counts,
                    i=j,
                    j=i,
                    denominator=endpoint_denominator_ij,
                    endpoint_threshold=endpoint_threshold,
                )

                consensus[i, j] = endpoint_i
                consensus[j, i] = endpoint_j

        consensus_matrices.append(consensus)

    return consensus_matrices

In [33]:
consensus_matrices = build_consensus_matrices(
    summaries=bootstrap_summaries,
    adjacency_threshold=0.70,
    endpoint_threshold=0.60,
    endpoint_denominator="all_graphs",
)

In [34]:
consensus_stage_1 = consensus_matrices[0]
consensus_stage_2 = consensus_matrices[1]
consensus_stage_3 = consensus_matrices[2]
consensus_stage_4 = consensus_matrices[3]

In [35]:
GRAPHVIZ_ENDPOINTS = {
    TAIL: "none",
    ARROW: "normal",
    CIRCLE: "odot",

    # Graphviz does not have a causal-learn STAR endpoint.
    # Diamond is used as a distinguishable approximation.
    STAR: "diamond",

    # Composite Graphviz endpoint markers.
    TAIL_AND_ARROW: "teenormal",
    ARROW_AND_ARROW: "normalnormal",
}

In [36]:
def draw_consensus_matrix(
    consensus_matrix: np.ndarray,
    feature_names: Sequence[str],
    positions: Dict[str, tuple],
    adjacency_counts: np.ndarray = None,
    n_bootstraps: int = None,
    show_adjacency_frequency: bool = True,
    width: str = "100%",
    max_width: int = 950,
    max_height: int = 700,
):
    """
    Draw a bootstrap consensus graph in the original Wille positions.
    """
    names = [
        str(name).strip().upper()
        for name in feature_names
    ]

    matrix = np.asarray(
        consensus_matrix,
        dtype=int,
    )

    expected_shape = (
        len(names),
        len(names),
    )

    if matrix.shape != expected_shape:
        raise ValueError(
            f"Expected matrix shape {expected_shape}, "
            f"received {matrix.shape}."
        )

    missing_positions = [
        name
        for name in names
        if name not in positions
    ]

    if missing_positions:
        raise ValueError(
            "No fixed position for: "
            + ", ".join(missing_positions)
        )

    if show_adjacency_frequency:
        if adjacency_counts is None or n_bootstraps is None:
            raise ValueError(
                "adjacency_counts and n_bootstraps are required "
                "when show_adjacency_frequency=True."
            )

        adjacency_counts = np.asarray(
            adjacency_counts,
            dtype=int,
        )

        if adjacency_counts.shape != expected_shape:
            raise ValueError(
                "adjacency_counts has an incompatible shape."
            )

    graph = pydot.Dot(
        graph_type="graph",
        strict=False,
        overlap="true",
        splines="true",
        outputorder="edgesfirst",
        margin="0",
        pad="0.15",
    )

    # --------------------------------------------------------------
    # Nodes
    # --------------------------------------------------------------
    for index, name in enumerate(names):
        x, y = positions[name]

        node = pydot.Node(
            str(index),
            label=name,
            pos=f"{x},{y}!",
            pin="true",
            shape="box",
            style="rounded,filled",
            fillcolor="white",
            fontsize="10",
            margin="0.07,0.04",
            penwidth="1",
        )

        graph.add_node(node)

    # --------------------------------------------------------------
    # Edges
    # --------------------------------------------------------------
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            endpoint_i = int(matrix[i, j])
            endpoint_j = int(matrix[j, i])

            if endpoint_i == NULL and endpoint_j == NULL:
                continue

            if endpoint_i == NULL or endpoint_j == NULL:
                raise ValueError(
                    f"Invalid one-sided adjacency between "
                    f"{names[i]} and {names[j]}."
                )

            edge_options = {
                "dir": "both",
                "arrowtail": GRAPHVIZ_ENDPOINTS[endpoint_i],
                "arrowhead": GRAPHVIZ_ENDPOINTS[endpoint_j],
                "penwidth": "1.2",
            }

            if show_adjacency_frequency:
                frequency = (
                    adjacency_counts[i, j]
                    / n_bootstraps
                )

                edge_options["label"] = f"{frequency:.2f}"
                edge_options["fontsize"] = "8"

            edge = pydot.Edge(
                str(i),
                str(j),
                **edge_options,
            )

            graph.add_edge(edge)

    # --------------------------------------------------------------
    # Invisible anchors preserve the complete paper layout
    # --------------------------------------------------------------
    anchor_positions = {
        "__bottom_left": (-0.3, 0.3),
        "__bottom_right": (11.0, 0.3),
        "__top_left": (-0.3, 9.9),
        "__top_right": (11.0, 9.9),
    }

    for anchor_name, (x, y) in anchor_positions.items():
        graph.add_node(
            pydot.Node(
                anchor_name,
                label="",
                shape="point",
                width="0",
                height="0",
                style="invis",
                pos=f"{x},{y}!",
                pin="true",
            )
        )

    svg = graph.create_svg(
        prog="neato"
    ).decode("utf-8")

    svg = svg.replace(
        "<svg ",
        (
            f'<svg style="'
            f'width:{width}; '
            f'max-width:{max_width}px; '
            f'height:auto;'
            f'" '
        ),
        1,
    )

    display(
        HTML(
            f"""
            <div style="
                width: 100%;
                max-height: {max_height}px;
                overflow: auto;
                text-align: center;
            ">
                {svg}
            </div>
            """
        )
    )

In [37]:
draw_consensus_matrix(
    consensus_matrix=consensus_matrices[0],
    feature_names=bootstrap_summaries[0].feature_names,
    positions=WILLE_POSITIONS,
    adjacency_counts=bootstrap_summaries[0].adjacency_counts,
    n_bootstraps=bootstrap_summaries[0].n_bootstraps,
)

In [38]:
def draw_all_consensus_graphs(
    consensus_matrices: Sequence[np.ndarray],
    summaries: Sequence[BootstrapStageSummary],
    positions: Dict[str, tuple],
    show_adjacency_frequency: bool = True,
):
    if len(consensus_matrices) != len(summaries):
        raise ValueError(
            "The number of consensus matrices and summaries differs."
        )

    for stage_id, (matrix, summary) in enumerate(
        zip(consensus_matrices, summaries)
    ):
        display(
            HTML(
                f"<h3>Incremental stage {stage_id + 1}</h3>"
            )
        )

        draw_consensus_matrix(
            consensus_matrix=matrix,
            feature_names=summary.feature_names,
            positions=positions,
            adjacency_counts=summary.adjacency_counts,
            n_bootstraps=summary.n_bootstraps,
            show_adjacency_frequency=show_adjacency_frequency,
        )

In [39]:
draw_all_consensus_graphs(
    consensus_matrices=consensus_matrices,
    summaries=bootstrap_summaries,
    positions=WILLE_POSITIONS,
    show_adjacency_frequency=True,
)